# Cattle Weight — Step E: Ensemble (B0 + S + Mobile)
Attach datasets: `cow-data-prep`, `cow-effnetb0`, `cow-effnets`, `cow-mobile`.
Loads the three trained models and averages their predictions.

In [ ]:
import subprocess, sys
def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)
pip('albumentations', 'scikit-image', 'rembg[cpu]')

In [ ]:
import os, json, warnings, math, tarfile
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm.auto import tqdm
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
import albumentations as A
warnings.filterwarnings('ignore')

print('TF  :', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
tf.keras.mixed_precision.set_global_policy('mixed_float16')

SEED = 42
OUT  = Path('/kaggle/working')
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# ── Load shared data-prep outputs ─────────────────────────────────────────────
def _find_data_prep():
    for p in sorted(Path('/kaggle/input').rglob('norm_state.json')):
        return str(p.parent)
    raise FileNotFoundError(
        "norm_state.json not found under /kaggle/input. "
        "Make sure you attached the cow-data-prep dataset to this kernel.")

DATA_PREP = _find_data_prep()
print(f'Data prep found at: {DATA_PREP}')

state       = json.load(open(f'{DATA_PREP}/norm_state.json'))
LOG_MEAN    = state['log_mean'];    LOG_STD     = state['log_std']
WEIGHT_MEAN = state['weight_mean']; WEIGHT_STD  = state['weight_std']
print(f'Norm state: log_mean={LOG_MEAN:.4f}  log_std={LOG_STD:.4f}')

train_df = pd.read_csv(f'{DATA_PREP}/train_split.csv')
val_df   = pd.read_csv(f'{DATA_PREP}/val_split.csv')
test_df  = pd.read_csv(f'{DATA_PREP}/test_split.csv')
for df in [train_df, val_df, test_df]:
    df['bbox'] = df['bbox'].apply(
        lambda x: json.loads(x) if pd.notna(x) and str(x) not in ('nan','null','') else None)
print(f'Split: train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}')

def _find_dataset_root(sample_path):
    slug = Path(sample_path).parts[3]
    for root, dirs, _ in os.walk('/kaggle/input'):
        if slug in dirs:
            return os.path.join(root, slug)
    return None

def _remap_paths(df):
    sample = df['full_path'].dropna().iloc[0]
    if Path(sample).exists():
        return df
    old_root = '/'.join(Path(sample).parts[:4])
    new_root = _find_dataset_root(sample)
    if new_root:
        df = df.copy()
        df['full_path'] = df['full_path'].str.replace(old_root, new_root, regex=False)
        print(f'Remapped: {old_root} -> {new_root}')
    else:
        print(f'[WARN] Could not find dataset root for: {sample}')
    return df

train_df = _remap_paths(train_df)
val_df   = _remap_paths(val_df)
test_df  = _remap_paths(test_df)
_s = train_df['full_path'].dropna().iloc[0]
print(f'Path check: {"OK" if Path(_s).exists() else "BROKEN"} — {_s[:80]}')

MASK_DIR = Path('/kaggle/working/masks')
MASK_DIR.mkdir(exist_ok=True)
_tar_hits = (list(Path('/kaggle/input').rglob('masks.tar.gz'))
             or list(Path('/kaggle/input').rglob('masks.tar')))
if _tar_hits:
    _tar = _tar_hits[0]
    print(f'Extracting masks from {_tar} ...')
    with tarfile.open(_tar) as t:
        t.extractall('/kaggle/working/')
    print(f'  {sum(1 for _ in MASK_DIR.glob("*.png")):,} masks ready')
else:
    _mask_dirs = [p for p in Path('/kaggle/input').rglob('masks') if p.is_dir()]
    if _mask_dirs:
        import shutil
        _all_pngs = [p for _md in _mask_dirs for p in _md.rglob('*.png')]
        print(f'  Found {len(_all_pngs)} PNG masks under {_mask_dirs[0]}')
        for _png in _all_pngs:
            shutil.copy2(str(_png), str(MASK_DIR / _png.name))

_mask_count = sum(1 for _ in MASK_DIR.glob('*.png'))
if _mask_count > 0:
    print(f'  {_mask_count:,} masks ready')
else:
    print('Masks empty/missing — generating with rembg...')
    try:
        from rembg import remove, new_session
        _sess = new_session('u2net')
        _all = (pd.concat([train_df, val_df, test_df], ignore_index=True)
                  [['full_path', 'file_name']].drop_duplicates())
        def _gen_mask(row):
            try:
                stem = Path(row['file_name']).stem
                out  = MASK_DIR / f'{stem}.png'
                if out.exists(): return
                mask = remove(Image.open(row['full_path']).convert('RGB'),
                              session=_sess, only_mask=True)
                mask.save(out)
            except Exception: pass
        for _row in tqdm(_all.to_dict('records'), desc='rembg masks'):
            _gen_mask(_row)
        print(f'  Generated {sum(1 for _ in MASK_DIR.glob("*.png")):,} masks')
    except BaseException as _e:
        print(f'[WARN] rembg failed ({_e}) — body-geometry features will be zeros')

In [ ]:
# ── Preprocessing pipeline (identical to training notebooks) ──────────────────
IMG_SIZE = 224
_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]

val_aug = A.Compose([
    A.LongestMaxSize(IMG_SIZE),
    A.PadIfNeeded(IMG_SIZE, IMG_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0),
    A.Normalize(_MEAN, _STD),
])
AUX_DIM = 16

class GeM(layers.Layer):
    def __init__(self, p=3.0, eps=1e-6, **kw):
        super().__init__(**kw)
        self._p  = tf.Variable(p, trainable=True, dtype=tf.float32, name='gem_p')
        self.eps = eps
    def call(self, x):
        x = tf.cast(x, tf.float32)
        x = tf.pow(tf.maximum(x, self.eps), self._p)
        x = tf.reduce_mean(x, axis=[1, 2])
        return tf.cast(tf.pow(x, 1.0 / self._p), self.compute_dtype)
    def get_config(self):
        return {**super().get_config(), 'p': float(self._p.numpy()), 'eps': self.eps}

def _extract_from_array(mask_crop):
    try:
        H, W = mask_crop.shape[:2]
        _, binary = cv2.threshold(mask_crop, 127, 255, cv2.THRESH_BINARY)
        total_px = H * W
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            return np.zeros(14, np.float32)
        cnt = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(cnt)
        perimeter = cv2.arcLength(cnt, True) + 1e-6
        hull_area = cv2.contourArea(cv2.convexHull(cnt)) + 1e-6
        x, y, bw, bh = cv2.boundingRect(cnt)
        M = cv2.moments(cnt)
        cx = M['m10'] / (M['m00'] + 1e-6) / W
        cy = M['m01'] / (M['m00'] + 1e-6) / H
        roi = cv2.resize(binary[y:y+bh, x:x+bw], (128, 128)) > 127
        try:
            from skimage.morphology import skeletonize
            skel_len = float(skeletonize(roi).sum()) / (128 * 128)
        except Exception:
            skel_len = 0.0
        diag = np.sqrt(W**2 + H**2) + 1e-6
        ma, mi, angle = (0.0, 0.0, 0.0)
        if len(cnt) >= 5:
            _, (ma, mi), angle = cv2.fitEllipse(cnt)
        return np.clip(np.array([
            area / total_px, bw / (bh + 1e-6), bw / W, bh / H,
            area / hull_area, (4 * np.pi * area) / (perimeter ** 2),
            area / (bw * bh + 1e-6), skel_len,
            ma / diag, mi / diag,
            np.sin(np.deg2rad(angle)), np.cos(np.deg2rad(angle)),
            cx, cy,
        ], np.float32), 0.0, 1.0)
    except Exception:
        return np.zeros(14, np.float32)

CACHE_SIZE = 256
IMG_CACHE  = {}

def _preload_row(row):
    path = row.get('full_path', '')
    if not path or path in IMG_CACHE:
        return
    img = cv2.imread(path)
    if img is None:
        IMG_CACHE[path] = (np.zeros((CACHE_SIZE, CACHE_SIZE, 3), np.uint8),
                           np.zeros(14, np.float32))
        return
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    H0, W0 = img.shape[:2]
    crop_coords = None
    bbox = row.get('bbox')
    if isinstance(bbox, list):
        try:
            x, y, bw, bh = [max(0, int(v)) for v in bbox]
            pad = int(max(bw, bh) * 0.12)
            x1=max(0,x-pad); y1=max(0,y-pad)
            x2=min(W0,x+bw+pad); y2=min(H0,y+bh+pad)
            if x2 > x1 and y2 > y1:
                crop_coords = (x1, y1, x2, y2)
                img = img[y1:y2, x1:x2]
        except Exception:
            pass
    stem = Path(row.get('file_name', '')).stem
    mask_path = MASK_DIR / f'{stem}.png'
    body_feats = np.zeros(14, np.float32)
    if mask_path.exists():
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is not None:
            mc = mask[crop_coords[1]:crop_coords[3],
                      crop_coords[0]:crop_coords[2]] if crop_coords else mask
            body_feats = _extract_from_array(mc)
            if mc.shape[:2] == img.shape[:2]:
                img = (img * (mc[:,:,None] / 255.0)).astype(np.uint8)
    h, w = img.shape[:2]
    s = CACHE_SIZE / max(h, w, 1)
    nh, nw = max(1, int(h*s)), max(1, int(w*s))
    img = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    img = cv2.copyMakeBorder(img, 0, CACHE_SIZE-nh, 0, CACHE_SIZE-nw,
                             cv2.BORDER_CONSTANT, value=0)
    IMG_CACHE[path] = (img.astype(np.uint8), body_feats)

print('Preloading images...')
_all_rows = (pd.concat([train_df, val_df, test_df], ignore_index=True)
               .drop_duplicates('full_path').to_dict('records'))
for _row in tqdm(_all_rows, desc='preloading'):
    _preload_row(_row)
_cache_mb = sum(img.nbytes for img,_ in IMG_CACHE.values()) / 1e6
print(f'Cached {len(IMG_CACHE):,} images ({_cache_mb:.0f} MB)')

def load_sample(row):
    _entry = IMG_CACHE.get(row.get('full_path', ''))
    if _entry is not None:
        img, body_feats = _entry[0].copy(), _entry[1].copy()
    else:
        img = np.zeros((CACHE_SIZE, CACHE_SIZE, 3), np.uint8)
        body_feats = np.zeros(14, np.float32)
    img = val_aug(image=img)['image'].astype(np.float32)
    age   = row.get('age', float('nan'))
    age_n = float(age) / 10.0 if pd.notna(age) else 0.0
    sex_b = 1.0 if str(row.get('sex', '')).lower() in ('m', 'male', 'bull') else 0.0
    aux   = np.concatenate([[age_n, sex_b], body_feats]).astype(np.float32)
    log_w = np.log(float(row['weight_kg']))
    return img, aux, np.float32((log_w - LOG_MEAN) / LOG_STD)

def make_infer_ds(df_split, batch_size=128):
    rows = df_split.to_dict('records')
    def tf_load(idx):
        img, aux, label = tf.py_function(
            lambda i: load_sample(rows[int(i)]), [idx],
            [tf.float32, tf.float32, tf.float32])
        img.set_shape((IMG_SIZE, IMG_SIZE, 3))
        aux.set_shape((AUX_DIM,))
        label.set_shape(())
        return {'image': img, 'aux': aux}, label
    ds = tf.data.Dataset.from_tensor_slices(np.arange(len(rows)))
    return (ds.map(tf_load, num_parallel_calls=tf.data.AUTOTUNE)
              .cache()
              .batch(batch_size)
              .prefetch(tf.data.AUTOTUNE))

BATCH_SIZE = 128
val_ds  = make_infer_ds(val_df,  BATCH_SIZE)
test_ds = make_infer_ds(test_df, BATCH_SIZE)
print('Inference datasets ready')

In [ ]:
# ── Load models ───────────────────────────────────────────────────────────────
custom_objects = {'GeM': GeM}

def find_model(pattern):
    hits = list(Path('/kaggle/input').rglob(pattern))
    if not hits:
        raise FileNotFoundError(f'{pattern} not found under /kaggle/input')
    return str(hits[0])

model_paths = [
    ('B0',     find_model('effnetv2b0_best.keras')),
    ('S',      find_model('effnetv2s_best.keras')),
    ('Mobile', find_model('mobilenetv3_best.keras')),
]

models = []
for name, path in model_paths:
    print(f'Loading {name} from {path}')
    m = keras.models.load_model(path, custom_objects=custom_objects, compile=False)
    models.append((name, m))
    print(f'  {name} loaded — {sum(p.numpy().size for p in m.trainable_weights):,} trainable params')

# ── Collect predictions ───────────────────────────────────────────────────────
def get_preds_and_labels(ds):
    all_labels = []
    for _, labels in ds:
        all_labels.append(labels.numpy())
    return np.concatenate(all_labels)

def predict_all(ds, models):
    preds = {}
    for name, m in models:
        print(f'  predicting {name}...')
        p = m.predict(ds, verbose=0)
        preds[name] = p.flatten()
    return preds

def compute_mape_mae(pred_norm, true_norm):
    pred_kg = np.exp(pred_norm * LOG_STD + LOG_MEAN)
    true_kg = np.exp(true_norm * LOG_STD + LOG_MEAN)
    mape = float(np.mean(np.abs(pred_kg - true_kg) / (true_kg + 1e-6)) * 100)
    mae  = float(np.mean(np.abs(pred_kg - true_kg)))
    return mape, mae

# ── Val set: individual + ensemble ───────────────────────────────────────────
print('\n--- Val set ---')
val_labels = get_preds_and_labels(val_ds)
val_preds  = predict_all(val_ds, models)

for name, p in val_preds.items():
    mape, mae = compute_mape_mae(p, val_labels)
    print(f'  {name:8s}  MAPE={mape:.2f}%  MAE={mae:.1f} kg')

# Simple average in normalised log-weight space
val_ensemble = np.mean(list(val_preds.values()), axis=0)
mape_e, mae_e = compute_mape_mae(val_ensemble, val_labels)
print(f'  Ensemble  MAPE={mape_e:.2f}%  MAE={mae_e:.1f} kg')

# ── Test set: individual + ensemble ──────────────────────────────────────────
print('\n--- Test set ---')
test_labels = get_preds_and_labels(test_ds)
test_preds  = predict_all(test_ds, models)

for name, p in test_preds.items():
    mape, mae = compute_mape_mae(p, test_labels)
    print(f'  {name:8s}  MAPE={mape:.2f}%  MAE={mae:.1f} kg')

test_ensemble = np.mean(list(test_preds.values()), axis=0)
mape_e, mae_e = compute_mape_mae(test_ensemble, test_labels)
print(f'  Ensemble  MAPE={mape_e:.2f}%  MAE={mae_e:.1f} kg')

# ── Save results ──────────────────────────────────────────────────────────────
results = {
    'val': {
        **{n: dict(zip(['mape','mae_kg'], compute_mape_mae(p, val_labels)))
           for n, p in val_preds.items()},
        'ensemble': {'mape': float(mape_e), 'mae_kg': float(mae_e)},
    },
    'test': {
        **{n: dict(zip(['mape','mae_kg'], compute_mape_mae(p, test_labels)))
           for n, p in test_preds.items()},
        'ensemble': {},
    },
}
# overwrite test ensemble with correct values
mape_t, mae_t = compute_mape_mae(test_ensemble, test_labels)
results['test']['ensemble'] = {'mape': mape_t, 'mae_kg': mae_t}
json.dump(results, open(OUT / 'ensemble_results.json', 'w'), indent=2)
print(f'\nSaved ensemble_results.json')
print(f'Final ensemble test MAPE={mape_t:.2f}%  MAE={mae_t:.1f} kg')